# Multi-Class Classification Matrix

## Multiclass Confusion Matrix: Definition and Notation

A **multiclass confusion matrix** compares the true class of every observation with the class predicted by a classifier. For a problem with $K$ classes, define

$$
C_{ij}=\#\{n:y_n=i\text{ and }\hat{y}_n=j\},\qquad i,j\in\{1,\ldots,K\}.
$$

Here, $C_{ij}$ is the number of observations whose **actual** class is $i$ and whose **predicted** class is $j$. Rows correspond to actual classes and columns correspond to predicted classes. The diagonal entries are correct predictions; off-diagonal entries are misclassifications. The total number of observations is

$$
N=\sum_{i=1}^{K}\sum_{j=1}^{K}C_{ij}.
$$

This notebook uses the Iris classes setosa, versicolor, and virginica. In scikit-learn, `confusion_matrix(y_true, y_pred)` places actual labels on rows and predicted labels on columns; unless `labels` is specified, the class labels are ordered. Confirm the label order before interpreting a matrix.

## One-vs-Rest Counts for a Class

### Deriving Metrics for One Class (One-vs-Rest)

Multiclass precision, recall, and F1-score are calculated separately for each class by treating that class as **positive** and combining all remaining classes into one **negative** group. This is the **one-vs-rest** approach. For a chosen class $i$, rows are actual classes and columns are predicted classes, so:

$$
\begin{aligned}
TP_i &= C_{ii},\\
FN_i &= \sum_{j\ne i} C_{ij},\\
FP_i &= \sum_{j\ne i} C_{ji},\\
TN_i &= \sum_{j\ne i}\sum_{k\ne i} C_{jk}.
\end{aligned}
$$

The **false negative** sum uses row $i$: the actual class is $i$, but the prediction is some other class $j$. The **false positive** sum uses column $i$: the prediction is $i$, but the actual class is some other class $j$. The **true negative** double sum includes entries where both the actual class $j$ and predicted class $k$ are different from $i$. These sums partition the matrix into the four one-vs-rest outcomes.

The four counts can also be read from row and column totals:

| Count | Meaning for class $k$ | Formula from $C$ |
|---|---|---|
| True positive ($TP_k$) | Actual class $k$, predicted class $k$ | $C_{kk}$ |
| False negative ($FN_k$) | Actual class $k$, predicted as another class | $\sum_j C_{kj}-C_{kk}$ (row $k$, excluding diagonal) |
| False positive ($FP_k$) | Another actual class, predicted as class $k$ | $\sum_i C_{ik}-C_{kk}$ (column $k$, excluding diagonal) |
| True negative ($TN_k$) | Neither actual nor predicted class $k$ | $N-TP_k-FN_k-FP_k$ |

The $TN_k$ formula follows because every observation belongs to exactly one of these four disjoint groups; subtract the other three groups from the total $N$.

## Classification Metrics: Definitions and Derivations

### Accuracy

**Definition:** the proportion of all observations classified correctly. Correct classifications are exactly the diagonal entries of $C$, so

$$
\mathrm{Accuracy}=\frac{\text{number correct}}{\text{number of observations}}
=\frac{\sum_{k=1}^{K}C_{kk}}{N}.
$$

### Precision for class $k$

**Definition:** among observations predicted as class $k$, the fraction that truly belongs to class $k$. All predictions of class $k$ are in column $k$, whose sum is $TP_k+FP_k$. Therefore,

$$
P_k=\frac{TP_k}{TP_k+FP_k}
=\frac{C_{kk}}{\sum_i C_{ik}}.
$$

### Recall (sensitivity) for class $k$

**Definition:** among observations that truly belong to class $k$, the fraction predicted correctly. All actual class-$k$ observations are in row $k$, whose sum is $TP_k+FN_k$. Therefore,

$$
R_k=\frac{TP_k}{TP_k+FN_k}
=\frac{C_{kk}}{\sum_j C_{kj}}.
$$

### Specificity for class $k$

**Definition:** among observations that do not belong to class $k$, the fraction correctly predicted as not class $k$. The actual negatives consist of $TN_k+FP_k$, so

$$
S_k=\frac{TN_k}{TN_k+FP_k}.
$$

### F1-score for class $k$

**Definition:** the harmonic mean of precision and recall. Starting with $F1_k=2P_kR_k/(P_k+R_k)$ and substituting $P_k=TP_k/(TP_k+FP_k)$ and $R_k=TP_k/(TP_k+FN_k)$ gives

$$
\begin{aligned}
F1_k&=\frac{2\,\frac{TP_k}{TP_k+FP_k}\,\frac{TP_k}{TP_k+FN_k}}
{\frac{TP_k}{TP_k+FP_k}+\frac{TP_k}{TP_k+FN_k}}\\
&=\frac{2TP_k}{2TP_k+FP_k+FN_k}.
\end{aligned}
$$

The harmonic mean penalizes a low precision or recall: a high value of only one does not produce a high F1-score. Compute each class's F1 first, then average those scores when using macro or weighted F1.

## Averaging Metrics Across Classes

Precision, recall, F1, and specificity are first defined per class. An averaging rule combines the $K$ class scores $M_k$ (where $M$ is the metric):

- **Macro average:** $M_{\mathrm{macro}}=\frac{1}{K}\sum_{k=1}^{K}M_k$. Every class has equal weight, regardless of its support.
- **Weighted average:** $M_{\mathrm{weighted}}=\frac{\sum_{k=1}^{K}s_kM_k}{N}$, where $s_k=\sum_j C_{kj}$ is the number of actual examples (support) of class $k$. Classes with more actual examples contribute more.
- **Micro average:** sum the class-wise counts first, then calculate the metric. For example, $P_{\mathrm{micro}}=\frac{\sum_kTP_k}{\sum_k(TP_k+FP_k)}$; micro recall and F1 are calculated analogously from pooled counts.

For **single-label multiclass classification**, each observation has one true label and one predicted label. In that setting, micro precision, micro recall, and micro F1 are all equal to accuracy. Macro averaging is useful when each class should matter equally; weighted averaging reflects class frequencies in the evaluated data. If a denominator is zero, that per-class metric is undefined; scikit-learn's `zero_division` option controls the convention (the default warns and assigns zero).

In [1]:
import numpy as np
import pandas as pd

# importing Iris Dataset
from sklearn.datasets import load_iris

In [2]:
df = load_iris()
df

{'data': array([[5.1, 3.5, 1.4, 0.2],
        [4.9, 3. , 1.4, 0.2],
        [4.7, 3.2, 1.3, 0.2],
        [4.6, 3.1, 1.5, 0.2],
        [5. , 3.6, 1.4, 0.2],
        [5.4, 3.9, 1.7, 0.4],
        [4.6, 3.4, 1.4, 0.3],
        [5. , 3.4, 1.5, 0.2],
        [4.4, 2.9, 1.4, 0.2],
        [4.9, 3.1, 1.5, 0.1],
        [5.4, 3.7, 1.5, 0.2],
        [4.8, 3.4, 1.6, 0.2],
        [4.8, 3. , 1.4, 0.1],
        [4.3, 3. , 1.1, 0.1],
        [5.8, 4. , 1.2, 0.2],
        [5.7, 4.4, 1.5, 0.4],
        [5.4, 3.9, 1.3, 0.4],
        [5.1, 3.5, 1.4, 0.3],
        [5.7, 3.8, 1.7, 0.3],
        [5.1, 3.8, 1.5, 0.3],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.7, 1.5, 0.4],
        [4.6, 3.6, 1. , 0.2],
        [5.1, 3.3, 1.7, 0.5],
        [4.8, 3.4, 1.9, 0.2],
        [5. , 3. , 1.6, 0.2],
        [5. , 3.4, 1.6, 0.4],
        [5.2, 3.5, 1.5, 0.2],
        [5.2, 3.4, 1.4, 0.2],
        [4.7, 3.2, 1.6, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5.4, 3.4, 1.5, 0.4],
        [5.2, 4.1, 1.5, 0.1],
  

In [3]:
#  Converting Iris Dataset to DataFrame Using Pandas
df = pd.DataFrame(load_iris().data, columns=load_iris().feature_names)
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [4]:
# Giving ID to first column
df.insert(0, 'ID', range(1, len(df) + 1))
df.head()

,ID,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,1,5.1,3.5,1.4,0.2
1,2,4.9,3.0,1.4,0.2
2,3,4.7,3.2,1.3,0.2
3,4,4.6,3.1,1.5,0.2
4,5,5.0,3.6,1.4,0.2


In [5]:
from sklearn.preprocessing import LabelEncoder
encoder= LabelEncoder()
# adding Species column wether its 0=iris-setosa, 1=iris-versicolor, or 2=iris-virginica
df['species'] = encoder.fit_transform(load_iris().target)
df

,ID,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,1,5.1,3.5,1.4,0.2,0
1,2,4.9,3.0,1.4,0.2,0
2,3,4.7,3.2,1.3,0.2,0
3,4,4.6,3.1,1.5,0.2,0
4,5,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,2
146,147,6.3,2.5,5.0,1.9,2
147,148,6.5,3.0,5.2,2.0,2
148,149,6.2,3.4,5.4,2.3,2


In [6]:
from sklearn.model_selection import train_test_split
X= df.iloc[:, 0:-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [8]:
clf_logistic_regression = LogisticRegression()
clf_decision_tree = DecisionTreeClassifier()

In [9]:
clf_logistic_regression.fit(X_train, y_train)
clf_decision_tree.fit(X_train, y_train)

/Users/I578366/Desktop/Data Science/Machine Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of sampl

In [10]:
y_pred_logistic_regression = clf_logistic_regression.predict(X_test)
y_pred_decision_tree = clf_decision_tree.predict(X_test)

In [11]:
from sklearn.metrics import confusion_matrix, accuracy_score

# Accuracy of the logistic regression classifier
accuracy_logistic_regression = accuracy_score(y_test, y_pred_logistic_regression)
print(accuracy_logistic_regression)

# Accuracy of the decision tree classifier
accuracy_decision_tree = accuracy_score(y_test, y_pred_decision_tree)
print(accuracy_decision_tree)

1.0
0.9666666666666667


In [13]:
print("Logistic Regression Confusion Matrix")
print(confusion_matrix(y_test, y_pred_logistic_regression))

Logistic Regression Confusion Matrix
[[11  0  0]
 [ 0 13  0]
 [ 0  0  6]]


In [14]:
print("Decision Tree Confusion Matrix")
print(confusion_matrix(y_test, y_pred_decision_tree))

Decision Tree Confusion Matrix
[[11  0  0]
 [ 0 12  1]
 [ 0  0  6]]


In [16]:
result = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Confusion Matrix": [confusion_matrix(y_test, y_pred_logistic_regression),
                         confusion_matrix(y_test, y_pred_decision_tree)]
})
print(result)

                 Model                     Confusion Matrix
0  Logistic Regression  [[11, 0, 0], [0, 13, 0], [0, 0, 6]]
1        Decision Tree  [[11, 0, 0], [0, 12, 1], [0, 0, 6]]


In [18]:
result = pd.DataFrame()
result['Actual Label'] = y_test
result['Logistic Regression Prediction'] = y_pred_logistic_regression
result['Decision Tree Prediction'] = y_pred_decision_tree
result.sample(10)

,Actual Label,Logistic Regression Prediction,Decision Tree Prediction
78,1,1,1
16,0,0,0
98,1,1,1
56,1,1,1
84,1,1,1
125,2,2,2
5,0,0,0
33,0,0,0
75,1,1,1
94,1,1,1


### Macro, Weighted, and Micro Averages

In this notebook, `confusion_matrix(y_test, y_pred)` does **not** perform macro or micro averaging; it returns the raw class-by-class counts. `accuracy_score` also does not take an `average` option: it returns the fraction of correct predictions. The `precision_score`, `recall_score`, and `f1_score` calls below explicitly set `average='macro'`, so they calculate each class's score separately and take the unweighted mean. They are **not** calculating micro averages. The weighted average instead weights each class score by its number of true examples (support). The micro average pools the class-wise true positives, false positives, and false negatives before computing the metric. In a single-label multiclass problem, micro precision, micro recall, and micro F1 equal accuracy.

Use macro averaging when all classes should count equally; use weighted averaging when the class frequencies in the test set should determine each class's contribution.

### Macro-Averaged Formulas

First calculate the metric for each of the $K$ classes, then take the unweighted mean:

$$
P_{\mathrm{macro}}=\frac{P_1+P_2+\cdots+P_K}{K},\qquad
R_{\mathrm{macro}}=\frac{R_1+R_2+\cdots+R_K}{K},
$$
$$
F1_{\mathrm{macro}}=\frac{F1_1+F1_2+\cdots+F1_K}{K}.
$$

### Micro-Averaged Formulas

First sum the one-vs-rest counts across all classes, then calculate the metric:

$$
P_{\mathrm{micro}}=\frac{\sum_{k=1}^{K}TP_k}{\sum_{k=1}^{K}(TP_k+FP_k)},\qquad
R_{\mathrm{micro}}=\frac{\sum_{k=1}^{K}TP_k}{\sum_{k=1}^{K}(TP_k+FN_k)},
$$
$$
F1_{\mathrm{micro}}=\frac{2\sum_{k=1}^{K}TP_k}{2\sum_{k=1}^{K}TP_k+\sum_{k=1}^{K}FP_k+\sum_{k=1}^{K}FN_k}.
$$

### Quick Comparison

| Aspect | Macro averaging | Micro averaging |
|---|---|---|
| Calculation | Average the per-class metric values. | Pool class-wise counts, then calculate the metric. |
| Class contribution | Every class has equal weight. | Classes contribute according to their counts. |
| Single-label Iris task | Unweighted mean across the three species. | Precision, recall, and F1 each equal accuracy. |

In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score
print("Logistic Regression Precision:", precision_score(y_test, y_pred_logistic_regression, average='macro'))
print("Logistic Regression Recall:", recall_score(y_test, y_pred_logistic_regression, average='macro'))
print("Logistic Regression F1 Score:", f1_score(y_test, y_pred_logistic_regression, average='macro'))

print("Decision Tree Precision:", precision_score(y_test, y_pred_decision_tree, average='macro'))
print("Decision Tree Recall:", recall_score(y_test, y_pred_decision_tree, average='macro'))
print("Decision Tree F1 Score:", f1_score(y_test, y_pred_decision_tree, average='macro'))

Logistic Regression Precision: 1.0
Logistic Regression Recall: 1.0
Logistic Regression F1 Score: 1.0
Decision Tree Precision: 0.9523809523809524
Decision Tree Recall: 0.9743589743589745
Decision Tree F1 Score: 0.9610256410256411


In [20]:
# Micro-averaged metrics pool the class-wise counts before calculating each score.
print("Logistic Regression Micro Precision:", precision_score(y_test, y_pred_logistic_regression, average='micro'))
print("Logistic Regression Micro Recall:", recall_score(y_test, y_pred_logistic_regression, average='micro'))
print("Logistic Regression Micro F1 Score:", f1_score(y_test, y_pred_logistic_regression, average='micro'))

print("Decision Tree Micro Precision:", precision_score(y_test, y_pred_decision_tree, average='micro'))
print("Decision Tree Micro Recall:", recall_score(y_test, y_pred_decision_tree, average='micro'))
print("Decision Tree Micro F1 Score:", f1_score(y_test, y_pred_decision_tree, average='micro'))

Logistic Regression Micro Precision: 1.0
Logistic Regression Micro Recall: 1.0
Logistic Regression Micro F1 Score: 1.0
Decision Tree Micro Precision: 0.9666666666666667
Decision Tree Micro Recall: 0.9666666666666667
Decision Tree Micro F1 Score: 0.9666666666666667


Micro averaging gives every individual observation equal importance. For a single-label multiclass classification problem, each observation has exactly one true class and one predicted class. Therefore, the total number of micro false positives equals the total number of micro false negatives, and:

$$
\mathrm{Precision}_{\mathrm{micro}}=\mathrm{Recall}_{\mathrm{micro}}=\mathrm{F1}_{\mathrm{micro}}=\mathrm{Accuracy}.
$$

Use macro averaging when class-level fairness is important, because it gives each class equal weight. Use micro averaging when you want a metric pooled across all individual predictions.

## Summary of Definitions and Formulas

| Quantity | Definition | Formula |
|---|---|---|
| Confusion-matrix entry | Count with actual class $i$ and predicted class $j$ | $C_{ij}=\#\{n:y_n=i,\hat{y}_n=j\}$ |
| True positive for class $k$ | Actual $k$, predicted $k$ | $TP_k=C_{kk}$ |
| False negative for class $k$ | Actual $k$, predicted not $k$ | $FN_k=\sum_jC_{kj}-C_{kk}$ |
| False positive for class $k$ | Actual not $k$, predicted $k$ | $FP_k=\sum_iC_{ik}-C_{kk}$ |
| True negative for class $k$ | Actual not $k$, predicted not $k$ | $TN_k=N-TP_k-FN_k-FP_k$ |
| Accuracy | Overall fraction correct | $\frac{\sum_kC_{kk}}{N}$ |
| Precision (class $k$) | Correct predictions among predictions of $k$ | $\frac{TP_k}{TP_k+FP_k}$ |
| Recall / sensitivity (class $k$) | Correct predictions among actual examples of $k$ | $\frac{TP_k}{TP_k+FN_k}$ |
| Specificity (class $k$) | Correct rejections among actual non-$k$ examples | $\frac{TN_k}{TN_k+FP_k}$ |
| F1-score (class $k$) | Harmonic mean of precision and recall | $\frac{2P_kR_k}{P_k+R_k}=\frac{2TP_k}{2TP_k+FP_k+FN_k}$ |
| Macro average | Unweighted mean of per-class metric | $\frac{1}{K}\sum_kM_k$ |
| Weighted average | Mean weighted by actual class support $s_k$ | $\frac{\sum_ks_kM_k}{N}$ |
| Micro average | Pool class-wise counts before computing metric | For precision: $\frac{\sum_kTP_k}{\sum_k(TP_k+FP_k)}$ |

**Key to reading the matrix:** rows are actual classes; columns are predicted classes. For single-label multiclass tasks, micro precision, micro recall, micro F1, and accuracy are equal.